# PipeGuard: QLoRA fine-tuning of gpt-oss-20b

Distils the production teacher (`openai/gpt-oss-120b` on Groq) into a LoRA adapter on
`gpt-oss-20b` for the **Monitoring** and **Remediation** agents, then evaluates the
fine-tuned model against the base model on held-out functions.

**Kaggle setup**
1. *Settings → Accelerator*: **GPU T4 x2** (training uses one T4, 16 GB). *Internet*: **on**.
2. *Add Input*: a Kaggle dataset named `pipeguard-sft` holding `train.jsonl` and `val.jsonl`
   from `backend/training/data/sft/` (produced by `python -m training.export_sft`).
3. *Run All*. The adapter and the evaluation results are written to `/kaggle/working/`.

In [ ]:
%%capture
!pip install --upgrade unsloth

In [ ]:
DATA_DIR = "/kaggle/input/pipeguard-sft"
MODEL_NAME = "unsloth/gpt-oss-20b"
MAX_SEQ_LENGTH = 4096        # longest prompt + answer; longer examples are dropped
EPOCHS = 2
MAX_EVAL = 20                # held-out examples per model; generation on a T4 is slow
OUTPUT_DIR = "/kaggle/working/pipeguard-gpt-oss-20b-lora"

## 1. Load gpt-oss-20b in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 2. Data

Each example is the exact production prompt (system + user) and the teacher's verified
answer. `reasoning_effort="low"` matches how the student will be called: it learns to
answer directly instead of spending tokens on hidden reasoning.

In [ ]:
import json
from collections import Counter
from datasets import Dataset

def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def render(messages, add_generation_prompt=False):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt, reasoning_effort="low",
    )

train_rows = read_jsonl(f"{DATA_DIR}/train.jsonl")
val_rows = read_jsonl(f"{DATA_DIR}/val.jsonl")

texts = [render(row["messages"]) for row in train_rows]
fits = [len(tokenizer(text)["input_ids"]) <= MAX_SEQ_LENGTH for text in texts]
train = Dataset.from_list([{"text": text} for text, ok in zip(texts, fits) if ok])

print("train:", len(train), "of", len(train_rows), Counter(r["meta"]["agent"] for r in train_rows))
print("val:  ", len(val_rows), Counter(r["meta"]["agent"] for r in val_rows))
print(train[0]["text"][:1200])

## 3. Train (loss on the assistant's answer only)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        seed=3407,
        output_dir="/kaggle/working/checkpoints",
        report_to="none",
    ),
)
# Mask the prompt: only the final-channel answer contributes to the loss.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>",
)

In [ ]:
import torch

stats = trainer.train()
print(f"training time: {stats.metrics['train_runtime'] / 60:.1f} min")
print(f"peak GPU memory: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

## 4. Evaluate on held-out functions

The validation split holds out whole functions, so these are bugs in code the model never
saw. Each output is checked the same way the generator labelled the teacher:

| Agent | `format_ok` | `grounded` |
|---|---|---|
| Monitoring | *Failed stage* + *Exact error* sections and `AGENT_CONFIDENCE` | names a failing test |
| Remediation | parsable steps JSON with at least 2 steps and `AGENT_CONFIDENCE` | names the buggy function |

In [ ]:
import re
import time

STEP_START, STEP_END = "===STEPS_JSON_START===", "===STEPS_JSON_END==="
FINAL = "<|channel|>final<|message|>"

def final_answer(raw):
    text = raw.split(FINAL)[-1] if FINAL in raw else raw
    for marker in ("<|return|>", "<|end|>"):
        text = text.split(marker)[0]
    return text.strip()

def check(agent, output, meta):
    has_confidence = re.search(r"AGENT_CONFIDENCE:\s*\d{1,3}", output) is not None
    if agent == "monitoring":
        tests = [test.split("::")[-1] for test in meta.get("failing_tests", [])]
        return {
            "format_ok": "**Failed stage**" in output and "**Exact error**" in output and has_confidence,
            "grounded": any(test in output for test in tests),
        }
    steps = []
    if STEP_START in output:
        block = output.split(STEP_START, 1)[1].split(STEP_END, 1)[0]
        try:
            steps = json.loads(block[block.find("["):block.rfind("]") + 1])
        except ValueError:
            steps = []
    return {
        "format_ok": len(steps) >= 2 and has_confidence,
        "grounded": meta.get("function", "") in output,
    }

def generate(messages, max_new_tokens=1500):
    prompt = render(messages, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    started = time.perf_counter()
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return {
        "text": final_answer(tokenizer.decode(new_tokens, skip_special_tokens=False)),
        "new_tokens": len(new_tokens),
        "seconds": time.perf_counter() - started,
    }

def evaluate(label, rows):
    results = []
    for row in rows:
        prompt = row["messages"][:-1]
        generated = generate(prompt)
        results.append({
            "model": label,
            "agent": row["meta"]["agent"],
            "group": row["meta"].get("group"),
            **check(row["meta"]["agent"], generated["text"], row["meta"]),
            "new_tokens": generated["new_tokens"],
            "seconds": round(generated["seconds"], 2),
            "output": generated["text"],
        })
        print(label, row["meta"]["agent"], {k: results[-1][k] for k in ("format_ok", "grounded", "new_tokens")})
    return results

In [ ]:
FastLanguageModel.for_inference(model)
eval_rows = val_rows[:MAX_EVAL]

results = evaluate("fine-tuned", eval_rows)
with model.disable_adapter():
    results += evaluate("base", eval_rows)

# Reference: the teacher's own (verified) answers on the same examples.
for row in eval_rows:
    answer = row["messages"][-1]["content"]
    results.append({
        "model": "teacher (gpt-oss-120b)",
        "agent": row["meta"]["agent"],
        "group": row["meta"].get("group"),
        **check(row["meta"]["agent"], answer, row["meta"]),
        "new_tokens": len(tokenizer(answer)["input_ids"]),  # visible answer only, excludes its hidden reasoning
        "seconds": None,
    })

In [ ]:
import pandas as pd

table = pd.DataFrame(results)
summary = table.groupby(["model", "agent"]).agg(
    examples=("format_ok", "size"),
    format_ok=("format_ok", "mean"),
    grounded=("grounded", "mean"),
    mean_new_tokens=("new_tokens", "mean"),
    mean_seconds=("seconds", "mean"),
).round(2)
summary

## 5. Save the adapter and the results

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
table.to_json("/kaggle/working/eval_results.jsonl", orient="records", lines=True)
summary.to_csv("/kaggle/working/eval_summary.csv")
!cd /kaggle/working && zip -qr pipeguard-gpt-oss-20b-lora.zip pipeguard-gpt-oss-20b-lora
print("Download from the Output tab: pipeguard-gpt-oss-20b-lora.zip, eval_summary.csv, eval_results.jsonl")

Optional: publish the adapter to the Hugging Face Hub (add an `HF_TOKEN` Kaggle secret first)
so it can be served with vLLM or a hosted LoRA endpoint:

```python
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
model.push_to_hub("<your-hf-user>/pipeguard-gpt-oss-20b-lora", token=token)
tokenizer.push_to_hub("<your-hf-user>/pipeguard-gpt-oss-20b-lora", token=token)
```